# Module 2: Tối ưu có ràng buộc và điều kiện KKT — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## Bảng ca KKT bằng sympy (dạng A)

min (x1−3)²+(x2−2)² với g1 = x1²+x2²−5 ≤ 0, g2 = x1+2x2−4 ≤ 0: giải từng ca rồi kiểm khả thi và λ ≥ 0.

In [1]:
import sympy as sp
x1, x2, l1, l2 = sp.symbols("x1 x2 l1 l2", real=True)
g1, g2 = x1**2 + x2**2 - 5, x1 + 2*x2 - 4
L = (x1 - 3)**2 + (x2 - 2)**2 + l1*g1 + l2*g2
stat = [sp.diff(L, x1), sp.diff(L, x2)]
cases = {"không": ({l1: 0, l2: 0}, []), "g1": ({l2: 0}, [g1]), "g2": ({l1: 0}, [g2]), "cả hai": ({}, [g1, g2])}
for name, (fix, eqs) in cases.items():
    vs = [v for v in (x1, x2, l1, l2) if v not in fix]
    for s in sp.solve([e.subs(fix) for e in stat] + eqs, vs, dict=True):
        s = {**fix, **s}
        ok = g1.subs(s) <= 1e-9 and g2.subs(s) <= 1e-9 and s[l1] >= 0 and s[l2] >= 0
        print(name, {k: round(float(v), 4) for k, v in s.items()}, "-> KKT" if ok else "-> loại")

không {l1: 0.0, l2: 0.0, x1: 3.0, x2: 2.0} -> loại
g1 {l2: 0.0, l1: 0.6125, x1: 1.8605, x2: 1.2403} -> loại
g1 {l2: 0.0, l1: -2.6125, x1: -1.8605, x2: -1.2403} -> loại
g2 {l1: 0.0, l2: 1.2, x1: 2.4, x2: 0.8} -> loại
cả hai {l1: -2.3333, l2: 4.9333, x1: -0.4, x2: 2.2} -> loại
cả hai {l1: 0.3333, l2: 0.6667, x1: 2.0, x2: 1.0} -> KKT


## Cùng bài bằng cvxpy (nhân tử dual_value)

cvxpy tự giải bài lồi và trả nhân tử Lagrange.

In [2]:
import cvxpy as cp
x = cp.Variable(2)
cons = [cp.sum_squares(x) <= 5, x[0] + 2*x[1] <= 4]
p = cp.Problem(cp.Minimize(cp.sum_squares(x - [3, 2])), cons); p.solve()
print("x* =", x.value.round(4), " f* =", round(p.value, 4))
print("lambda =", [round(float(c.dual_value), 4) for c in cons])

x* = [2. 1.]  f* = 2.0
lambda = [0.3333, 0.6667]


/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_87942/2163306859.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("lambda =", [round(float(c.dual_value), 4) for c in cons])


## Đẳng thức: nghiệm và nhân tử (dạng B)

min x1²+2x2² s.t. x1+x2=2.

In [3]:
import sympy as sp
x1, x2, mu = sp.symbols("x1 x2 mu")
L = x1**2 + 2*x2**2 + mu*(x1 + x2 - 2)
s = sp.solve([sp.diff(L, v) for v in (x1, x2, mu)], (x1, x2, mu))
print(s, " f* =", (x1**2 + 2*x2**2).subs(s))

{mu: -8/3, x1: 4/3, x2: 2/3}  f* = 8/3


## Bài code · Hàm kiểm điểm KKT

Đầu vào: $\nabla f$, danh sách $(g_i,\nabla g_i)$ tại điểm. Đầu ra: ràng buộc chặt, $\lambda$, phần dư, có phải KKT? (thử với $\bar x=(2,1)$ của bài ở tab Cách làm bài.)

In [4]:
import numpy as np
def check_kkt(gf, cons, tol=1e-7):
    act = [i for i, (g, _) in enumerate(cons) if abs(g) <= tol]
    M = np.column_stack([cons[i][1] for i in act]); lam = np.linalg.lstsq(M, -np.array(gf), rcond=None)[0]
    res = np.linalg.norm(M @ lam + gf)
    return act, lam, res, all(g <= tol for g, _ in cons) and res < 1e-6 and (lam >= -tol).all()
gf = [-2., -2.]; cons = [(0.0, [4., 2.]), (0.0, [1., 2.])]
act, lam, res, ok = check_kkt(gf, cons)
print("chặt:", [i + 1 for i in act], " lambda =", lam.round(4), " dư =", float(res) < 1e-6, " KKT:", bool(ok))

chặt: [1, 2]  lambda = [0.3333 0.6667]  dư = True  KKT: True


## Bài code · Độ nhạy: ý nghĩa của nhân tử

Đầu vào: $\min x^2$ s.t. $x\ge b$ (b = 1). Đầu ra: $p^*(b)$ tại $b=1$ và $1.01$; so độ dốc $dp^*/db$ với nhân tử $\lambda$ của ràng buộc $b-x\le0$.

In [5]:
import cvxpy as cp
def solve(b):
    x = cp.Variable(); c = [b - x <= 0]
    p = cp.Problem(cp.Minimize(x**2), c); p.solve(); return p.value, float(c[0].dual_value)
v1, lam = solve(1.0); v2, _ = solve(1.01)
print("lambda =", round(lam, 4), " (p*(1.01)-p*(1))/0.01 =", round((v2 - v1) / 0.01, 3))

lambda = 2.0  (p*(1.01)-p*(1))/0.01 = 2.01


## Bài code · scipy SLSQP vs cvxpy

Đầu ra: nghiệm của $\min(x_1-3)^2+(x_2-2)^2$ s.t. $x_1^2+x_2^2\le5,\ x_1+2x_2\le4$ bằng minimize; chú ý dấu ràng buộc.

In [6]:
import numpy as np
from scipy.optimize import minimize
f = lambda x: (x[0]-3)**2 + (x[1]-2)**2
cons = [{"type": "ineq", "fun": lambda x: 5 - x @ x}, {"type": "ineq", "fun": lambda x: 4 - x[0] - 2*x[1]}]
r = minimize(f, [0., 0.], constraints=cons, method="SLSQP", tol=1e-12); print(r.x.round(4), round(r.fun, 4))

[2. 1.] 2.0


## Bài code · Hàm đối ngẫu số

Đầu ra: với $\min x^2$ s.t. $x\ge1$: tính $g(\lambda)=\inf_xL$ trên lưới $\lambda$, tìm $\lambda^*$ và $d^*$.

In [7]:
import numpy as np
lam = np.linspace(0, 4, 401); g = lam - lam**2/4
print("lambda* =", lam[g.argmax()], " d* =", g.max(), "(= p* = 1)")

lambda* = 2.0  d* = 1.0 (= p* = 1)
